In [1]:
import re
import pandas as pd
from pathlib import Path
from datetime import datetime

import SCRIPTS.jsonDownloader as jd
from SCRIPTS.redditLinkRetriever import fetch_saved_post_links, save_links_txt
from SCRIPTS.mediaDownloader import download_embedded_media
from SCRIPTS.mediaOrganizer import organize_downloads
from SCRIPTS.redgifDownloader import process_external
from SCRIPTS.cloudflareUploader import upload_media
from SCRIPTS.r2_audit import audit_local_vs_r2

In [2]:
# https://old.reddit.com/prefs/apps
# Set to True to avoid making any changes
DRY_RUN_MEDIA = False
DRY_RUN_ORGANIZE = False

DRY_RUN_CLOUDFLARE = False   # True = preview only, no upload
ACCOUNT_INDEX = 2            # choose which R2 account (1, 2, ...)
CHECK_ONLY = False           # True = just check existence, no upload

DRY_RUN_FINAL = False

In [3]:
# Retrieve saved post links for the specified user
links = fetch_saved_post_links()

In [4]:
len(links)

237

In [5]:
links[:5]

['https://www.reddit.com/r/SanFranciscoNSFW/comments/1pspufu/public_sex_mission',
 'https://www.reddit.com/r/gettingbigger/comments/xzs8z7/how_to_get_hyperspermia',
 'https://www.reddit.com/r/SluttyConfessions/comments/1prp295/my_fwb_vs_my_hyperspermia_could_she_swallow_it_all',
 'https://www.reddit.com/r/cumsluts/comments/1ps8j91/somethings_gotta_blow',
 'https://www.reddit.com/r/rape_hentai/comments/1prxxi4/id_have_toys_of_all_my_friends_if_this_was']

# NEW POST VALIDATION

This section validates new posts from reddits saved folder

In [6]:
csv_path = Path("ordered_posts.csv")
raw_df = pd.read_csv(csv_path)

POST_ID_RE = re.compile(r"/comments/([a-z0-9]+)(?:[/?#]|$)", re.IGNORECASE)
SHORT_RE   = re.compile(r"redd\.it/([a-z0-9]+)(?:[/?#]|$)", re.IGNORECASE)
max_order_num = raw_df.order_num.max()

def strip_trailing_slash(url: str) -> str:
    # remove trailing slashes only at the very end (doesn't touch scheme)
    return url.rstrip("/")

def extract_post_id(url: str) -> str | None:
    """
    Try to extract a post id from:
      - standard permalink: .../comments/<postid>/...
      - shortlink: https://redd.it/<postid>
    """
    m = POST_ID_RE.search(url)
    if m:
        return m.group(1)
    m = SHORT_RE.search(url)
    if m:
        return m.group(1)
    return None

existing_ids = set(str(x).lower() for x in raw_df.get("post_id", pd.Series([])).dropna())

new_rows = []
next_order = max_order_num + 1
seen_in_batch = set()  # avoid duplicates within this run

for raw_link in reversed(links):
    link = strip_trailing_slash(raw_link)
    post_id = extract_post_id(link)
    if not post_id:
        continue
    pid = post_id.lower()

    # Only add if NOT already in CSV and not already queued this batch
    if pid in existing_ids or pid in seen_in_batch:
        continue

    new_rows.append({
        "order_num": next_order,
        "link": link,
        "post_id": post_id,
        "date_added": datetime.utcnow().isoformat(timespec="seconds"),
    })
    seen_in_batch.add(pid)
    next_order += 1

# Preview as a DataFrame
new_df = pd.DataFrame(new_rows)
new_df


C:\Users\minds\AppData\Local\Temp\ipykernel_21956\787629229.py:47: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "date_added": datetime.utcnow().isoformat(timespec="seconds"),


,order_num,link,post_id,date_added
0,1525,https://www.reddit.com/r/cumsluts/comments/1pp...,1ppzly1,2025-12-22T10:15:21
1,1526,https://www.reddit.com/r/NexxxtUp/comments/1pq...,1pq0icu,2025-12-22T10:15:21
2,1527,https://www.reddit.com/r/bangmybully/comments/...,1pq5tzk,2025-12-22T10:15:21
3,1528,https://www.reddit.com/r/SluttyConfessions/com...,1pqs1tt,2025-12-22T10:15:21
4,1529,https://www.reddit.com/r/bangmybully/comments/...,1pqp0b4,2025-12-22T10:15:21
5,1530,https://www.reddit.com/r/SluttyConfessions/com...,1prcx9x,2025-12-22T10:15:21
6,1531,https://www.reddit.com/r/bangmybully/comments/...,1pqrhv1,2025-12-22T10:15:21
7,1532,https://www.reddit.com/r/bangmybully/comments/...,1prgp2y,2025-12-22T10:15:21
8,1533,https://www.reddit.com/r/bangmybully/comments/...,1ps3sru,2025-12-22T10:15:21
9,1534,https://www.reddit.com/r/bdsm/comments/1pqymdw...,1pqymdw,2025-12-22T10:15:21


In [7]:
final_df = pd.concat([raw_df, new_df], ignore_index=True)
final_df = final_df.sort_values(by="order_num", ascending=False).reset_index(drop=True)

In [8]:
import importlib
importlib.reload(jd)

jd.configure(
    DATA_ROOT="Out",
    SKIP_EXISTING=False,
    REPORTS_DIR="__reports",
    write_csv_to=None
    )

summary = jd.process_all(new_df["link"].tolist(), show_progress=True)
summary

  0%|          | 0/18 [00:00<?, ?post/s]

Done. Success: 18, Skipped: 0, Failed: 0


{'success': 18, 'skipped': 0, 'failed': 0}

# MEDIA DOWNLOADER
Reviews the external and media json folders in **Out/**, downloading:
- Images
- Gifs
- Videos

In [9]:
folders = ["external", "media"]
download_stats = []

# point to your inputs/outputs explicitly
for mediaType in folders:
    download_stats.append(download_embedded_media(
        media_json_dir=Path("Out/" + mediaType),   # where your *.json live
        media_out_dir=Path("Media/media_files"),  # where downloads should go
        write_fail_csv_to=Path("__reports/media_report" + datetime.now().strftime("%Y%m%d-%H%M%S") + ".csv"),
        show_progress=True,
    ))

download_stats

[{'downloaded': 0,
  'failed': 9,
  'skipped': 0,
  'fail_rows': [{'id': '1ppzly1', 'reason': 'no_reddit_media_url'},
   {'id': '1pq0icu', 'reason': 'no_reddit_media_url'},
   {'id': '1pqp0b4', 'reason': 'no_reddit_media_url'},
   {'id': '1pqrhv1', 'reason': 'no_reddit_media_url'},
   {'id': '1pqymdw', 'reason': 'no_reddit_media_url'},
   {'id': '1prgp2y', 'reason': 'no_reddit_media_url'},
   {'id': '1ps3sru', 'reason': 'no_reddit_media_url'},
   {'id': '1ps619h', 'reason': 'no_reddit_media_url'},
   {'id': '1ps8j91', 'reason': 'no_reddit_media_url'}],
  'out_dir': WindowsPath('Media/media_files'),
  'json_dir': WindowsPath('Out/external')},
 {'downloaded': 6,
  'failed': 0,
  'skipped': 0,
  'fail_rows': [],
  'out_dir': WindowsPath('Media/media_files'),
  'json_dir': WindowsPath('Out/media')}]

In [10]:
move_stats = organize_downloads(
    input_dir="Media/media_files",  # where your downloader wrote files
    output_dir="Media",             # where Images/, Videos/, Gifs/ live
    strategy="move",
    conflict="move_existing",
    show_progress=True,
    dry_run=DRY_RUN_ORGANIZE,       # set True to preview
    prune_empty_galleries=True,     # remove empty src folders after moving
)

move_stats

Organizing media:   0%|          | 0/6 [00:00<?, ?file/s]

{'moved': 6,
 'copied': 0,
 'linked': 0,
 'skipped': 0,
 'unknown': 0,
 'dry_run': False,
 'strategy': 'move',
 'conflict': 'move_existing',
 'input_dir': 'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\media_files',
 'output_dir': 'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media',
 'errors': [],
 'created_dirs': {'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\Gifs',
  'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\Images',
  'S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\Images\\1pspufu'},
 'pruned_dirs': ['S:\\minds\\Desktop\\Downloader and Reddit System\\Saved-Reddit\\Media\\media_files\\1pspufu']}

# REDGIF DOWNLOADER

Downloads redgifs from external json folder in **Out/**

In [11]:
stats = process_external(
    media_json_dir=Path("Out/external"),
    media_out_dir=Path("Media/RedGiphys"),
    write_fail_csv_to=Path("__reports/redgif_report_" + datetime.now().strftime("%Y%m%d-%H%M%S") + ".csv"),
    write_links_csv_to=Path("__reports/external_links" + datetime.now().strftime("%Y%m%d-%H%M%S") + ".csv"),
    show_progress=True,
    dry_run=DRY_RUN_MEDIA,
    overwrite_downloads=False,
)

stats

Found 9 external post JSONs in Out\external


Scanning external posts:   0%|          | 0/9 [00:00<?, ?post/s]

[REDGIFS] id=1ppzly1 -> 1ppzly1.mp4
[REDGIFS] id=1pq0icu -> 1pq0icu.mp4
[REDGIFS] id=1pqp0b4 -> 1pqp0b4.mp4
[REDGIFS] id=1pqrhv1 -> 1pqrhv1.mp4
[REDGIFS] id=1pqymdw -> 1pqymdw.mp4
[REDGIFS] id=1prgp2y -> 1prgp2y.mp4
[REDGIFS] id=1ps3sru -> 1ps3sru.mp4
[REDGIFS] id=1ps619h -> 1ps619h.mp4
[REDGIFS] id=1ps8j91 -> 1ps8j91.mp4
Saved external links to: S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\__reports\external_links20251222-021534.csv


{'external_rows': [{'id': '1ppzly1',
   'link': 'https://v3.redgifs.com/watch/fancyoptimalrabidsquirrel',
   'domain': 'v3.redgifs.com'},
  {'id': '1pq0icu',
   'link': 'https://redgifs.com/watch/newexhaustedhalcyon',
   'domain': 'redgifs.com'},
  {'id': '1pqp0b4',
   'link': 'https://www.redgifs.com/watch/puzzledoldfashionediberianbarbel',
   'domain': 'www.redgifs.com'},
  {'id': '1pqrhv1',
   'link': 'https://www.redgifs.com/watch/offbeathairyastarte',
   'domain': 'www.redgifs.com'},
  {'id': '1pqymdw',
   'link': 'https://www.redgifs.com/watch/profitableburlywoodmoa',
   'domain': 'www.redgifs.com'},
  {'id': '1prgp2y',
   'link': 'https://www.redgifs.com/watch/scentedflawlesswrasse',
   'domain': 'www.redgifs.com'},
  {'id': '1ps3sru',
   'link': 'https://www.redgifs.com/watch/homelydefensiveracerunner',
   'domain': 'www.redgifs.com'},
  {'id': '1ps619h',
   'link': 'https://www.redgifs.com/watch/punydiscreteiberianchiffchaff',
   'domain': 'www.redgifs.com'},
  {'id': '1ps8j91

# CLOUDFLARE VERIFICATION & UPLOAD

In [12]:
raw_output = []
uploadsData = []

for mediaType in ["Images", "Videos", "Gifs", "RedGiphys"]:
    try:
        result = upload_media(
            input_path=Path("Media") / mediaType,  # where local files/galleries live
            r2_prefix=mediaType,                   # must match bucket prefix
            account_idx=ACCOUNT_INDEX - 1,         # <— choose which R2 credentials to use
            dry_run=DRY_RUN_CLOUDFLARE,            # preview vs. real upload
            overwrite=False,                       # don't overwrite existing objects
            check_only=CHECK_ONLY,                 # <— enable to just check existence
        )

        raw_output.append(result)
        # choose which list you want to visualize depending on mode
        if CHECK_ONLY:
            uploadsData.extend(result["exists"] + result["missing"])
        else:
            uploadsData.extend(result["planned"])

    except Exception as e:
        print(f"⚠️ Error on {mediaType}: {e}")
        continue

⚠️ Error on Videos: Input path not found or not a directory: S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\Videos


In [13]:
results_df = pd.DataFrame(uploadsData)
pd.set_option('display.max_rows', None)
results_df

,local,r2_key,bytes,content_type,status,account_index,bucket
0,S:\minds\Desktop\Downloader and Reddit System\...,Images/1prxxi4.jpeg,291120,image/jpeg,planned,1,media-archive
1,S:\minds\Desktop\Downloader and Reddit System\...,Images/1pspufu/01.jpg,96034,image/jpeg,planned,1,media-archive
2,S:\minds\Desktop\Downloader and Reddit System\...,Images/1pspufu/02.jpg,72086,image/jpeg,planned,1,media-archive
3,S:\minds\Desktop\Downloader and Reddit System\...,Images/1pspufu/03.jpg,115129,image/jpeg,planned,1,media-archive
4,S:\minds\Desktop\Downloader and Reddit System\...,Gifs/1pq5tzk.gif,43968718,image/gif,planned,1,media-archive
5,S:\minds\Desktop\Downloader and Reddit System\...,Gifs/1psi75t.gif,9420427,image/gif,planned,1,media-archive
6,S:\minds\Desktop\Downloader and Reddit System\...,RedGiphys/1ppzly1.mp4,40078881,video/mp4,planned,1,media-archive
7,S:\minds\Desktop\Downloader and Reddit System\...,RedGiphys/1pq0icu.mp4,16419273,video/mp4,planned,1,media-archive
8,S:\minds\Desktop\Downloader and Reddit System\...,RedGiphys/1pqp0b4.mp4,36152844,video/mp4,planned,1,media-archive
9,S:\minds\Desktop\Downloader and Reddit System\...,RedGiphys/1pqrhv1.mp4,48725092,video/mp4,planned,1,media-archive


# VERIFY UPLOAD

In [14]:
cats = ["Images", "RedGiphys", "Gifs", "Videos"]
all_rows = []

for cat in cats:
    try:
        res = audit_local_vs_r2(
            local_root=Path("Media") / cat,  # e.g., Media/Images
            r2_prefixes=[cat],               # matches your bucket key prefix
            account_indices=None,            # all accounts
            write_csv_to=None,               # (optional) per-cat CSV
            show_progress=True,
        )
        rows = res["rows"]
        for r in rows:
            r["category"] = cat
        all_rows.extend(rows)
    except Exception as e:
        print(f"⚠️ Error auditing {cat}: {e}")
        continue

audit_results = pd.DataFrame(all_rows)
pd.set_option("display.max_rows", None)
audit_results

Auditing: 100%|██████████| 2/2 [00:00<?, ?file/s]

⚠️ Error auditing Videos: Local images root not found or not a directory: S:\minds\Desktop\Downloader and Reddit System\Saved-Reddit\Media\Videos


,local_rel,local_ext,all_expected_keys,matched,match_type,matched_prefix,matched_key,remote_ext,same_ext,matched_account_index,matched_bucket,note,category
0,1prxxi4.jpeg,.jpeg,Images/1prxxi4.jpeg,True,exact,Images,Images/1prxxi4.jpeg,.jpeg,True,1,media-archive,exact_match,Images
1,1pspufu/01.jpg,.jpg,Images/1pspufu/01.jpg,True,exact,Images,Images/1pspufu/01.jpg,.jpg,True,1,media-archive,exact_match,Images
2,1pspufu/02.jpg,.jpg,Images/1pspufu/02.jpg,True,exact,Images,Images/1pspufu/02.jpg,.jpg,True,1,media-archive,exact_match,Images
3,1pspufu/03.jpg,.jpg,Images/1pspufu/03.jpg,True,exact,Images,Images/1pspufu/03.jpg,.jpg,True,1,media-archive,exact_match,Images
4,1ppzly1.mp4,.mp4,RedGiphys/1ppzly1.mp4,True,exact,RedGiphys,RedGiphys/1ppzly1.mp4,.mp4,True,1,media-archive,exact_match,RedGiphys
5,1pq0icu.mp4,.mp4,RedGiphys/1pq0icu.mp4,True,exact,RedGiphys,RedGiphys/1pq0icu.mp4,.mp4,True,1,media-archive,exact_match,RedGiphys
6,1pqp0b4.mp4,.mp4,RedGiphys/1pqp0b4.mp4,True,exact,RedGiphys,RedGiphys/1pqp0b4.mp4,.mp4,True,1,media-archive,exact_match,RedGiphys
7,1pqrhv1.mp4,.mp4,RedGiphys/1pqrhv1.mp4,True,exact,RedGiphys,RedGiphys/1pqrhv1.mp4,.mp4,True,1,media-archive,exact_match,RedGiphys
8,1pqymdw.mp4,.mp4,RedGiphys/1pqymdw.mp4,True,exact,RedGiphys,RedGiphys/1pqymdw.mp4,.mp4,True,1,media-archive,exact_match,RedGiphys
9,1prgp2y.mp4,.mp4,RedGiphys/1prgp2y.mp4,True,exact,RedGiphys,RedGiphys/1prgp2y.mp4,.mp4,True,1,media-archive,exact_match,RedGiphys


In [15]:
if DRY_RUN_FINAL or (False in audit_results["matched"].value_counts().keys()):
    print(final_df.head(20))
    print("Dry run enabled; ordered_posts not changed")
else:
    print("Updating ordered_posts.csv with new posts...")
    final_df.to_csv("ordered_posts.csv", index=False)

Updating ordered_posts.csv with new posts...
